# MandiLense — Advanced Data Mining & Machine Learning

## Academic Syllabus Mapping:
- **Supervised Classification**: Decision Tree Induction, Ensemble Bagging (Random Forest), Attribute Selection (Gini Impurity).
- **Supervised Regression**: Simple & Multiple Linear Regression, Error Measures (MAE, RMSE, R²).
- **Model Evaluation**: Precision, Recall, F1-Score, Confusion Matrix, ROC-AUC, Chronological Holdout (80/20).
- **Unsupervised Clustering**: K-Means Clustering, Elbow Method (Inertia), Silhouette Score Optimization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score
)

BASE_DATA = Path("../Data/processed/mining")

---
## 1. Classification: Agricultural Price Movement Direction
### Objective:
Classify commodity price movements into 3 discrete classes:
- **Increase**: Price rise > +5%
- **Stable**: Price fluctuation within [-5%, +5%]
- **Decrease**: Price drop < -5%

> **Anti-Data-Leakage Design**: We deliberately exclude `price_change_pct` and current `avg_modal_price` from the training features, as the ground-truth target label is derived from them. The model must learn purely from historical lag features, momentum, and geographic variance.

In [ ]:
df_class = pd.read_csv(BASE_DATA / "classification.csv")
print(f"Dataset shape: {df_class.shape}")
print("Class distribution:")
print(df_class["target"].value_counts())
df_class.head()

In [ ]:
TARGET = "target"
LEAKY = {"price_change_pct", "avg_modal_price", "target"}
features = [c for c in df_class.columns if c not in LEAKY]

# Chronological Holdout (80% train, 20% test)
df_class = df_class.sort_values(["year", "month"]).reset_index(drop=True)
split_idx = int(0.8 * len(df_class))

X_train, X_test = df_class[features].iloc[:split_idx], df_class[features].iloc[split_idx:]
y_train, y_test = df_class[TARGET].iloc[:split_idx], df_class[TARGET].iloc[split_idx:]

numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = [c for c in features if c not in numeric_cols]

# Pipeline with ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

pipeline = Pipeline([("prep", preprocessor), ("model", rf_model)])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

probas = pipeline.predict_proba(X_test)
auc = roc_auc_score(y_test, probas, multi_class="ovr")
print(f"Multiclass ROC-AUC (One-vs-Rest): {auc:.4f}")

In [ ]:
# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)
labels = pipeline.classes_

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix - Price Movement")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 Feature Importances
ohe_names = []
if categorical_cols:
    ohe = pipeline.named_steps["prep"].named_transformers_["cat"]
    ohe_names = list(ohe.get_feature_names_out(categorical_cols))
all_names = numeric_cols + ohe_names

importances = pd.Series(pipeline.named_steps["model"].feature_importances_, index=all_names).sort_values(ascending=False)

plt.figure(figsize=(9, 4))
importances.head(10).plot(kind="barh", color="#2e7d32").invert_yaxis()
plt.title("Top 10 Feature Importances (Gini Impurity Metric)")
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()

---
## 2. Regression: Next-Month Modal Price Prediction
### Objective:
Estimate the exact next month price using Linear Regression: $y = w^T X + b$.

In [ ]:
df_reg = pd.read_csv(BASE_DATA / "regression.csv").dropna().reset_index(drop=True)
df_reg = df_reg.sort_values(["year", "month"]).reset_index(drop=True)

TARGET_REG = "target_price"
reg_features = [c for c in df_reg.columns if c != TARGET_REG]

split_r = int(0.8 * len(df_reg))
Xr_train, Xr_test = df_reg[reg_features].iloc[:split_r], df_reg[reg_features].iloc[split_r:]
yr_train, yr_test = df_reg[TARGET_REG].iloc[:split_r], df_reg[TARGET_REG].iloc[split_r:]

scaler_reg = StandardScaler()
Xr_train_s = scaler_reg.fit_transform(Xr_train)
Xr_test_s = scaler_reg.transform(Xr_test)

lr = LinearRegression()
lr.fit(Xr_train_s, yr_train)
yr_pred = lr.predict(Xr_test_s)

mae = mean_absolute_error(yr_test, yr_pred)
rmse = mean_squared_error(yr_test, yr_pred) ** 0.5
r2 = r2_score(yr_test, yr_pred)

print("=== Regression Evaluation Metrics ===")
print(f"MAE : Rs {mae:,.2f}")
print(f"RMSE: Rs {rmse:,.2f}")
print(f"R2  : {r2:.4f}")

---
## 3. Unsupervised Clustering: Mandi Market Segmentation (K-Means)
### Objective:
Segment agricultural commodities into distinct risk/profile clusters based on `avg_price`, `volatility`, and `mean_change`.

In [ ]:
df_clust = pd.read_csv(BASE_DATA / "clustering_labeled.csv")
print("Cluster Counts:")
print(df_clust["cluster"].value_counts())

print("\nCluster Profile Summary (Means):")
display_cols = ["avg_price", "volatility", "mean_change", "num_months"]
print(df_clust.groupby("cluster")[display_cols].mean())

In [ ]:
# Visualizing Commodity Clusters on Log Scale
plt.figure(figsize=(8, 5))
colors = {0: "#1976d2", 1: "#d32f2f", 2: "#388e3c"}
names = {
    0: "Cluster 0: Core Agricultural Staples",
    1: "Cluster 1: High Volatility Anomalies",
    2: "Cluster 2: Ultra High-Value Exotic / Spices"
}

for c_id in sorted(df_clust["cluster"].unique()):
    subset = df_clust[df_clust["cluster"] == c_id]
    plt.scatter(
        subset["avg_price"],
        subset["volatility"],
        label=names.get(c_id, f"Cluster {c_id}"),
        color=colors.get(c_id, "gray"),
        alpha=0.7,
        edgecolors="w",
        s=60
    )

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Average Modal Price (Log Scale Rs/Quintal)")
plt.ylabel("Price Volatility StdDev (Log Scale %)")
plt.title("Mandi Market Clustering (K-Means, k=3)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()